# Figure 3: Reference Frame Preferences

**Paper**: Parodi et al. (2026)

This notebook reproduces Figure 3, which characterizes the spatial reference frame preference of mid-STS neurons across all 5 sessions (1404 units, 2 subjects).

Each unit's encoding model is fit in three reference frames:
- **Allocentric (Allo)**: Position and movement in world coordinates
- **Body-centric (Body)**: Kinematics relative to the body axis
- **Head-centric (Head)**: Kinematics relative to head orientation

The **RF index** for each unit quantifies the relative preference: values near +1 indicate allo-dominant, near -1 indicate ego-dominant.

**Key finding**: ~60% of units are allo-dominant, with M2 showing stronger allocentric preference (74%) than M1 (27%).

In [ ]:
import sys, warnings
from pathlib import Path

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)

REPO_ROOT = Path("..").resolve()
sys.path.insert(0, str(REPO_ROOT))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

plt.rcParams.update({
    'font.size': 12, 'axes.labelsize': 14, 'axes.titlesize': 14,
    'xtick.labelsize': 11, 'ytick.labelsize': 11, 'legend.fontsize': 11,
    'figure.dpi': 150, 'axes.spines.top': False, 'axes.spines.right': False,
})

SESSION_RENAME = {
    "hedy_0526": "M2_0526", "hedy_0605": "M2_0605",
    "hooke_0525": "M1_0525", "hooke_0609": "M1_0609", "hooke_0613": "M1_0613",
}
SUBJECT_MAP = {"M2_0526": "M2", "M2_0605": "M2", "M1_0525": "M1", "M1_0609": "M1", "M1_0613": "M1"}
FRAME_COLORS = {"allo": "#457B9D", "body": "#E63946", "head": "#F4A261"}

print("Setup complete.")

## Load Production Results

In [ ]:
# Load Fig 3 production results
fig3_df = pd.read_parquet(REPO_ROOT / "results" / "production" / "fig3_bin05" / "summary" / "fig3_summary.parquet")
fig3_df["session_id"] = fig3_df["session_id"].map(SESSION_RENAME)
fig3_df["subject"] = fig3_df["session_id"].map(SUBJECT_MAP)

# Compute RF winner per unit
frames = ["allo", "body", "head"]
for f in frames:
    if f"unique_{f}_shuffle" not in fig3_df.columns:
        fig3_df[f"unique_{f}_shuffle"] = fig3_df.get(f"unique_{f}", 0)

fig3_df["winner"] = fig3_df[[f"unique_{f}_shuffle" for f in frames]].idxmax(axis=1).str.replace("unique_", "").str.replace("_shuffle", "")

print(f"Fig 3 data: {len(fig3_df)} units, {fig3_df['session_id'].nunique()} sessions")
print(f"\nWinner distribution:")
print(fig3_df["winner"].value_counts().to_string())
print(f"\nOverall: Allo={fig3_df['winner'].eq('allo').mean():.1%}, Body={fig3_df['winner'].eq('body').mean():.1%}, Head={fig3_df['winner'].eq('head').mean():.1%}")

### Fig 3A: Winner Distribution by Session

In [ ]:
# Fig 3A: Stacked bar chart of RF winner per session
sessions = sorted(fig3_df["session_id"].unique())
fig, ax = plt.subplots(figsize=(10, 5))

x = np.arange(len(sessions))
width = 0.6
bottom = np.zeros(len(sessions))

for frame in frames:
    pcts = [fig3_df[fig3_df["session_id"] == s]["winner"].eq(frame).mean() * 100 for s in sessions]
    ax.bar(x, pcts, width, bottom=bottom, label=frame.capitalize(), color=FRAME_COLORS[frame])
    bottom += pcts

ax.set_xticks(x)
ax.set_xticklabels(sessions, rotation=30, ha="right")
ax.set_ylabel("% of Units")
ax.set_title("Fig 3A: RF Winner Distribution by Session")
ax.legend(title="Winner Frame")
ax.set_ylim(0, 105)
plt.tight_layout()
plt.show()

# Per-subject summary
for subj in ["M2", "M1"]:
    s = fig3_df[fig3_df["subject"] == subj]
    print(f"{subj}: n={len(s)}, Allo={s['winner'].eq('allo').mean():.1%}, "
          f"Body={s['winner'].eq('body').mean():.1%}, Head={s['winner'].eq('head').mean():.1%}")

### Fig 3B: Unique Variance Boxplots by Reference Frame

In [ ]:
# Fig 3B: Boxplots of unique variance by frame, split by subject
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, (subj, title) in zip(axes, [("M2", "M2 (843 units)"), ("M1", "M1 (561 units)")]):
    s = fig3_df[fig3_df["subject"] == subj]
    data = [s[f"unique_{f}_shuffle"].dropna().values for f in frames]
    bp = ax.boxplot(data, labels=[f.capitalize() for f in frames], patch_artist=True,
                    widths=0.6, showfliers=False, medianprops=dict(color="black", linewidth=2))
    for patch, f in zip(bp["boxes"], frames):
        patch.set_facecolor(FRAME_COLORS[f])
        patch.set_alpha(0.7)
    ax.set_ylabel("Unique Variance (ΔR²)")
    ax.set_title(title)
    ax.axhline(0, color="gray", linestyle="--", linewidth=0.5)

fig.suptitle("Fig 3B: Unique Variance by Reference Frame", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()